# Western Balkan Studies - LANDs plots

## Define RUN TAG

In [ ]:
RUN_ID = 'vre_low_2025_20260313'
config_name='config_WB6.yaml'

* load packages

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import RES.visuals as vis
from RES import utility as utils
from RES.hdf5_handler import DataHandler
import RES.utility as utlis
import RES.lands as lands

plt.style.use('../RES/visual_styles/elsevier.mplstyle')
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
cfg=utils.load_config(f'../config/{config_name}')
# run_id:str=cfg.get('Scenario').get('run_id')

sub_national_unit_tag:str=cfg.get('GADM').get('datafield_mapping').get('NAME_2',None) or cfg.get('GADM').get('datafield_mapping').get('NAME_1')
country_name:str=cfg.get('country')
country_kwd=country_name.replace(' ','')
CRS_m = cfg.get('default_CRS').get('meters')  # Default metric CRS
CRS_d = cfg.get('default_CRS').get('degrees')  # Default geographic CRS
regions=list(cfg.get('region_mapping').keys()) #'AL','BA','XK','ME','MK','RS'
vis_save_to_root=utils.ensure_path(f"../vis/{country_kwd}/{RUN_ID}/Combined_regions")

## Load Store

In [ ]:
 #All the regions should have RUN_ID results available
combined_store:dict[dict] = {}
utils.print_update(level=1,message=f"Loading data stores for {country_name} regions with RUN_ID: {RUN_ID} and Regions: {regions}")
for region in regions:
    country_dict = {}
    try:
        store = Path(f"../data/store/{country_kwd}/{RUN_ID}/resources_{country_kwd}_{region}_{RUN_ID}.h5")
        assert store.exists(), f"Store path doesn't exist: {store}"
        res_data = DataHandler(store, show_structure=False)
        country_dict['cells'] = res_data.from_store('cells')
        country_dict['boundary'] = res_data.from_store('boundary')
        country_dict['lines'] = res_data.from_store('lines')
        # country_dict['LandAvailability'] = res_data.from_store('LandAvailability')
        combined_store[region] = country_dict
        utils.print_update(level=2,message=f"✓ Loaded data for : {cfg.get('region_mapping').get(region).get('name') if cfg.get('region_mapping').get(region) else region}.") 
    except Exception as e:
        print(f"X Error with region {region}: {e}")
        continue

## Load All Cells

In [ ]:
all_cells_dict:dict[pd.DataFrame]=[combined_store[region]['cells'] for region in combined_store]
all_cells_gdf = gpd.GeoDataFrame(pd.concat(all_cells_dict, ignore_index=False), crs=all_cells_dict[0].crs)

# Load Test/Validation data

In [ ]:
existing_VREs_data_path=Path("../data/validation_data/existing_VREs_WB6.csv")
if existing_VREs_data_path.exists():
    existing_VREs=pd.read_csv(existing_VREs_data_path)
    existing_VREs_gdf=gpd.GeoDataFrame(existing_VREs,geometry=gpd.points_from_xy(existing_VREs.Longitude,existing_VREs.Latitude),crs="EPSG:4326")
    utils.print_update(level=1,message=f"✓ Loaded validation data for existing VREs from {existing_VREs_data_path}")
else:
    existing_VREs_gdf=None
    utils.print_warning(f"Validation data for existing VREs not found at {existing_VREs_data_path}")

# Lands

- Prepare Combined Regional boundary

In [ ]:
# Combine all region boundaries into a single GeoDataFrame
boundary_gdfs = [combined_store[region]['boundary'] for region in combined_store]
all_boundary_gdf = gpd.GeoDataFrame(pd.concat(boundary_gdfs, ignore_index=True), crs=boundary_gdfs[0].crs)
all_boundary_dissolved = all_boundary_gdf.dissolve(by="Country")[["geometry"]].reset_index()

- Process the boundary info for raster plotting

In [ ]:
all_boundary_plot=all_boundary_dissolved.to_crs(CRS_m)
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = all_boundary_plot.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

* Country Map (base)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

fig, ax = plt.subplots(figsize=(12, 7))
fig.suptitle(f" {country_name}", fontsize=16, fontweight='bold')

all_boundary_plot.plot(
    column="Country",
    categorical=True,
    edgecolor="black",
    linewidth=0.8,
    alpha=0.5,
    ax=ax
)

for _, row in all_boundary_plot.iterrows():
    point = row.geometry.representative_point()
    txt = ax.annotate(
        row["Country"],
        xy=(point.x, point.y),
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        color="black"
    )
    txt.set_path_effects([
        pe.withStroke(linewidth=2, foreground="white")
    ])

ax.set_axis_off()
plt.tight_layout()
plt.savefig(f"{vis_save_to_root}/{country_name}_map.png")

- Create cells' instance for plotting (CRS-m)

In [ ]:
if all_cells_gdf.crs != CRS_m:
    all_cells_plot = all_cells_gdf.to_crs(CRS_m)
else:
    all_cells_plot = all_cells_gdf

## Corine Land Cover

- Load CLC raster config to extract file name and other attributes

In [ ]:
for raster in cfg.get('CORINE').get('raster_types'):
    if raster['name']=='CORINE_land_cover':
        CLC_raster_cfg=raster

- Create a WB6 clipped (to geom) raster for plotting purpose. Our main workflow gives Country specific rasters.

 > Clip Raster to WB6 boundary (otherwise raster distribution will show wrong numbers)

In [ ]:
CLC_source_raster_path=Path(f"../{CLC_raster_cfg['root']}/{CLC_raster_cfg['raster']}")

 - Clip CLC to Wb6 | Method 1
   - Load complete CLC data, use rio clip with reprojected (CRS_m) boundary  

!!! METHOD 1: Provides error and missing classes due to clipping logic

In [ ]:
CLC_da=utils.get_raster_da(CLC_source_raster_path) 

In [ ]:
# Ensure same CRS
# if WB6_boundary_dissolved_reproj.crs != CLC_da.rio.crs:
#     WB6_boundary_dissolved_reproj = WB6_boundary_dissolved_reproj.to_crs(CLC_da.rio.crs)

# Clip raster to boundary
# CLC_raster_WB6_da = CLC_da.rio.clip(WB6_boundary_dissolved_reproj.geometry,
#                                     # WB6_boundary_dissolved_reproj.crs,
#                                     drop=False,
#                                     invert=False,
#                                     all_touched=True)

 - Clip CLC to Wb6 | Method 2 (if more alteration necessary)
   - More Control to alter the resolution and CRS 

In [ ]:
CLC_regional_raster_path = lands.clip_to_boundary_and_resample_raster(in_raster_config= CLC_raster_cfg,
                                           boundary_name=country_kwd,
                                           boundary=all_boundary_plot,
                                           CRS_meters=CRS_m,
                                           source_raster_path=CLC_source_raster_path,
                                           )

In [ ]:
CLC_raster_regional_da=utils.get_raster_da(CLC_regional_raster_path)

In [ ]:
utils.check_raster_classes(CLC_da,CLC_raster_regional_da,all_boundary_plot)

- Load Legends and Raster layers attributes from config

In [ ]:
CLC_legends=pd.read_csv("../data/legends/CLC_2018_legend.csv")
class_inclusion_layers:dict=cfg.get('CORINE').get('raster_types')[0]['class_inclusion']

- Check summary of existing sites

In [ ]:
existing_VREs_gdf_with_landcover,existing_VREs_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=CLC_raster_regional_da,
    legend_df=CLC_legends,
    class_col_name="CLC_landcover",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=existing_VREs_summary,
    class_col="CLC_landcover_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Land-Cover Class and Technology",
    figsize=(7, 3.5),
    wrap_width=25,
    fontsize=6,
    colors=["#d6851c", "#6f32e0"],
    save_to=vis_save_to_root/f"existing_VREs_by_landcover_{country_kwd}.png"
)


In [ ]:
LandCover_CLC_class_distribution=lands.plot_raster_class_distribution(CLC_raster_regional_da,
                                     CLC_legends,
                                     show=True,
                                     figsize=(8,5),
                                     save_path=vis_save_to_root/f'CLC_class_distribution_{country_kwd}.png',
                                     pct_threshold=0.2)

- Define layers for plotting

In [ ]:
layers_included:dict=CLC_raster_cfg['class_inclusion']

- Plot CLC + Existing VREs + Boundaries

In [ ]:
classes_to_plot=layers_included # None , plots all layers

if classes_to_plot is None:
    title = "CLC 2018 - All Layers"
    save_to_path = vis_save_to_root/"CLC_AllLayers.png"
else:
    title = f"CLC 2018 - {country_kwd} - suitable for New VRE Sites"
    save_to_path = vis_save_to_root/f"CLC_{country_kwd}_forNewVREsites.png"

fig1,ax1,save_to1=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=CLC_da,
    raster_legends=CLC_legends,
    classes_to_plot=classes_to_plot, #layers,                 # <- all classes
    boundary=all_boundary_plot,
    existing_VREs_gdf=existing_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=10.0,
    figsize=(16, 9),
    dpi=500,
    marker_highlight_width=3,
    title=title,
    output_path=save_to_path,
    legend_anchor=(.98, .99),
    legend_fontsize=10
)

utils.print_update(level=1,message=f"plot saved to: {save_to_path}")

## GAEZ

### Terrain

In [ ]:
GAEZ_terrain_raster_path='../data/downloaded_data/GAEZ/Rasters_in_use/LR/ter/slpmed30s.tif'
GAEZ_terrain_raster_legends=pd.read_csv("../data/legends/gaez_terrains_legend.csv")
GAEZ_terrain_cfg= next((item for item in cfg.get('GAEZ').get('raster_types') if 'terrain_resources' in item.get('name', '')), None)

- Load Raster Data as data-array

In [ ]:
GAEZ_terrain_raster_da = utils.get_raster_da(GAEZ_terrain_raster_path) 

- Clip Raster to WB6 boundary (otherwise raster distribution will show wrong numbers)

In [ ]:
GAEZ_terrain_raster_da_clipped=utils.get_raster_da(lands.clip_to_boundary_and_resample_raster(in_raster_config=           GAEZ_terrain_cfg,
                                           boundary_name=country_kwd,
                                           boundary=all_boundary_plot,
                                           CRS_meters=CRS_m,
                                           source_raster_path=GAEZ_terrain_raster_path,
                                           ))

In [ ]:
# Ensure same CRS
if all_boundary_plot.crs != GAEZ_terrain_raster_da.rio.crs:
    WB6_boundary_GAEZ = all_boundary_plot.to_crs(GAEZ_terrain_raster_da.rio.crs)
else:
    WB6_boundary_GAEZ = all_boundary_plot
# Clip raster to boundary
GAEZ_terrain_raster_da_clipped = GAEZ_terrain_raster_da.rio.clip(WB6_boundary_GAEZ.geometry, WB6_boundary_GAEZ.crs)

- Review class distributions

In [ ]:
lands.plot_raster_class_distribution(GAEZ_terrain_raster_da_clipped,
                                     GAEZ_terrain_raster_legends,
                                     show=True,
                                     figsize=(8,3),
                                     save_path=vis_save_to_root/f'GAEZ_terrains_class_distribution_{country_kwd}.png')


- Review layers mapping to existing VREs

In [ ]:
existing_VREs_gdf_with_terrains,terrains_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=GAEZ_terrain_raster_da_clipped,
    legend_df=GAEZ_terrain_raster_legends,
    class_col_name="GAEZ_terrain",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=terrains_summary,
    class_col="GAEZ_terrain_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Terrain Class and Technology",
    figsize=(6, 2),
    wrap_width=25,
    fontsize=6,
    colors=["#d6851c", "#6f32e0"],
    save_to=vis_save_to_root/f"existing_VREs_by_terrain_{country_kwd}.png"
)


- Define the layers to be plotted

In [ ]:
layers_excluded:dict=GAEZ_terrain_cfg['class_exclusion']

# Unique CLC codes in your clipped area
unique_classes = np.unique(GAEZ_terrain_raster_da_clipped.values[~np.isnan(GAEZ_terrain_raster_da_clipped.values)])

layers_included = {
    tech: [c for c in unique_classes if c not in layers_excluded[tech] and c != 0]
    for tech in ['solar', 'wind']
}

- Plot GAEZ Terrain with defined layers and existing VREs

In [ ]:
classes_to_plot = layers_included # layers_included #None , plots all layers

if classes_to_plot is None:
    GAEZ_terrain_save_to_path=vis_save_to_root/f"GAEZ_terrains_{country_kwd}_allLayers.png"
    title = "GAEZ Terrains - All Layers"
else:
# Update DOC contents (if needed)
    GAEZ_terrain_save_to_path=vis_save_to_root/f"GAEZ_terrains_{country_kwd}_forNewVREsites.png"
    title = "GAEZ Terrains - For New VRE Sites"

fig2,ax2,save_to2=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=GAEZ_terrain_raster_da_clipped,
    raster_legends=GAEZ_terrain_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=all_boundary_plot,
    existing_VREs_gdf=existing_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=5.0,
    legend_fontsize=11,
    title=title,
    output_path=GAEZ_terrain_save_to_path,
    legend_anchor=(.1, .4),
    marker_highlight_width=3,
)
utils.print_update(level=1,message=f"plot saved to: {GAEZ_terrain_save_to_path}")

### Exclusion

In [ ]:
GAEZ_exclusion_raster_path = "../data/downloaded_data/GAEZ/Rasters_in_use/LR/excl/exclusion_2017.tif"
GAEZ_exclusion_raster_legends=pd.read_csv("../data/legends/gaez_exclusion_legend.csv")
GAEZ_exclusion_cover_cfg= next((item for item in cfg.get('GAEZ').get('raster_types') if 'exclusion_areas' in item.get('name', '')), None)

- Load Raster Data as data-array

In [ ]:
GAEZ_exclusion_raster_data = utils.get_raster_da(GAEZ_exclusion_raster_path)

- Clip to WB6 Boundary

In [ ]:
# Ensure same CRS
if all_boundary_plot.crs != GAEZ_exclusion_raster_data.rio.crs:
    WB6_boundary_dissolved = all_boundary_plot.to_crs(GAEZ_exclusion_raster_data.rio.crs)

# Clip raster to boundary
GAEZ_exclusion_raster_data_clipped = GAEZ_exclusion_raster_data.rio.clip(WB6_boundary_dissolved.geometry, WB6_boundary_dissolved.crs)

- Plot Class Distribution

In [ ]:
lands.plot_raster_class_distribution(GAEZ_exclusion_raster_data_clipped,
                                     GAEZ_exclusion_raster_legends,
                                     show=True,
                                     figsize=(8,3),
                                     save_path=vis_save_to_root/f'GAEZ_Exclusion_class_distribution_{country_kwd}.png')

- Review existing sites and excluded lands

In [ ]:
existing_VREs_gdf_with_exclusions,exclusions_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=GAEZ_exclusion_raster_data_clipped,
    legend_df=GAEZ_exclusion_raster_legends,
    class_col_name="GAEZ_exclusion",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=exclusions_summary,
    class_col="GAEZ_exclusion_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Exclusion Class and Technology",
    figsize=(8, 2),
    wrap_width=25,
    fontsize=8,
    colors=["#d6851c", "#6f32e0"],
    save_to=vis_save_to_root/f"existing_VREs_by_exclusions_{country_kwd}.png"
)

- Define layers to be plotted

In [ ]:
layers_excluded:dict=GAEZ_exclusion_cover_cfg['class_exclusion']

# Unique GAEZ terrain codes in your clipped area
unique_classes = np.unique(GAEZ_exclusion_raster_data.values[~np.isnan(GAEZ_exclusion_raster_data.values)])

layers_included = {
    tech: [c for c in unique_classes if c not in layers_excluded[tech] and c != 0]
    for tech in ['solar', 'wind']
}

- Plot

In [ ]:
classes_to_plot = layers_included  #None , plots all layers

if classes_to_plot is None:
    GAEZ_exclusion_save_to_path=vis_save_to_root/f"GAEZ_exclusion_{country_kwd}_allLayers.png"
    title = "GAEZ Exclusions - All Layers"
else:
# Update DOC contents (if needed)
    GAEZ_exclusion_save_to_path=vis_save_to_root/f"GAEZ_exclusion_{country_kwd}_forNewVREsites.png"
    title = "GAEZ Exclusions - For New VRE Sites"

fig3,ax3,save_to3=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=GAEZ_exclusion_raster_data_clipped,
    raster_legends=GAEZ_exclusion_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=all_boundary_plot,
    existing_VREs_gdf=existing_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=5.0,
    figsize=(16, 9),
    dpi=500,
    legend_fontsize=12,
    title=title,
    output_path=GAEZ_exclusion_save_to_path,
    legend_anchor=(.1, .3),
    marker_highlight_width=3,
)
utils.print_update(level=1,message=f"plot saved to: {GAEZ_exclusion_save_to_path}")

# Grid

In [ ]:
all_lines=[combined_store[region]['lines'] for region in combined_store]
all_lines = gpd.GeoDataFrame(pd.concat(all_lines, ignore_index=True), crs=all_lines[0].crs)

In [ ]:
# Convert to numeric
all_lines['voltage_kv'] = pd.to_numeric(all_lines['voltage'], errors='coerce') / 1000

# Define voltage bins
bins = [0, 12, 25, 132, 220, float("inf")]
labels = ["<12 kV", "12–25 kV", "25–132 kV", "132–220 kV", "≥220 kV"]
all_lines['voltage_class'] = pd.cut(all_lines['voltage_kv'], bins=bins, labels=labels, right=False)


- Plot gird

In [ ]:
from matplotlib.patches import Patch
ax=all_boundary_gdf.plot(edgecolor='black', facecolor='grey', linewidth=0.2, figsize=(10, 8),alpha=0.1)

ax.set_axis_off()

if existing_VREs_gdf is not None and not existing_VREs_gdf.empty:

    existing_VREs_gdf[existing_VREs_gdf['Technology'].str.lower()=='wind'].plot(ax=ax, color='None', edgecolor='blue',markersize=60, linewidth=0.8,marker='o', label='Existing VREs',alpha=1,zorder=2)

    # Add legend for existing wind (purple)
    legend_handles = [Patch(facecolor='none', edgecolor='purple', label='Existing Wind', alpha=1)]
    ax.legend(handles=legend_handles, loc='upper left', fontsize=8, frameon=False)

    existing_VREs_gdf[existing_VREs_gdf['Technology'].str.lower()=='solar'].plot(ax=ax, color='None', edgecolor='orangered',markersize=60,linewidth=0.8, marker='o', label='Existing VREs',alpha=1,zorder=2)
all_lines.plot('voltage_class',ax=ax, figsize=(10, 8), legend=True)

plt.savefig(vis_save_to_root/f"WB6_existing_VREs_and_lines_{country_kwd}.png", dpi=500, bbox_inches='tight')